In [7]:
# Cell 1: Import Libs
import os, json, zipfile, glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
print('Libraries loaded')

Libraries loaded


In [8]:
# Cell 2: Dataset
zips = glob.glob('/content/*.zip')
if zips:
    with zipfile.ZipFile(zips[0], 'r') as z:
        z.extractall('/content/mlx90640_dataset')
    print(f'Extracted: {zips[0]}')
else:
    print('No zip found - make sure you uploaded the Kaggle dataset')

DATASET_PATH = '/content/mlx90640_dataset/thermal image'

def load_jpeg(path):
    img = Image.open(path).convert('L')
    img = img.resize((8, 8), Image.LANCZOS)
    return np.array(img, dtype=np.uint8)

def load_json(path):
    with open(path) as f:
        data = json.load(f)
    if isinstance(data, list):
        frame = np.array(data, dtype=np.float32).flatten()
    elif isinstance(data, dict):
        for key in ['frame', 'data', 'temperatures', 'pixels']:
            if key in data:
                frame = np.array(data[key], dtype=np.float32).flatten()
                break
        else:
            frame = np.array(list(data.values())[0], dtype=np.float32).flatten()
    else:
        return None
    if len(frame) == 768:          # 32x24 -> downsample to 8x8
        arr = frame.reshape(24, 32)
        out = np.zeros((8, 8))
        for r in range(8):
            for c in range(8):
                out[r, c] = arr[r*3:(r+1)*3, c*4:(c+1)*4].mean()
        frame = out.flatten()
    elif len(frame) != 64:
        return None
    mn, mx = frame.min(), frame.max()
    if mx > mn:
        frame = ((frame - mn) / (mx - mn) * 255).astype(np.uint8)
    else:
        frame = np.full(64, 128, dtype=np.uint8)
    return frame

real_X, real_y = [], []
for label_folder, label in [('human', 1), ('not human', 0)]:
    folder = os.path.join(DATASET_PATH, label_folder)
    if not os.path.exists(folder):
        print(f'Folder not found: {folder}')
        continue
    loaded = 0
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        frame = None
        try:
            if f.lower().endswith('.json'):
                frame = load_json(path)
            elif f.lower().endswith(('.jpeg', '.jpg', '.png')):
                frame = load_jpeg(path)
        except:
            pass
        if frame is not None:
            real_X.append(frame)
            real_y.append(label)
            loaded += 1
    print(f'{label_folder}: {loaded} frames')

print(f'Total real: {len(real_X)} ({sum(real_y)} human, {len(real_y)-sum(real_y)} not human)')

No zip found - make sure you uploaded the Kaggle dataset
Folder not found: /content/mlx90640_dataset/thermal image/human
Folder not found: /content/mlx90640_dataset/thermal image/not human
Total real: 0 (0 human, 0 not human)


In [9]:
# Cell 3: Clean Data and Training
real_X_clean = [x for x in real_X if len(x) == 64]
real_y_clean = [real_y[i] for i, x in enumerate(real_X) if len(x) == 64]
syn_X_clean  = [x for x in syn_X if len(x) == 64]
syn_y_clean  = [syn_y[i] for i, x in enumerate(syn_X) if len(x) == 64]

print(f'Real after filter:      {len(real_X_clean)} (removed {len(real_X)-len(real_X_clean)})')
print(f'Synthetic after filter: {len(syn_X_clean)} (removed {len(syn_X)-len(syn_X_clean)})')

X = np.array(real_X_clean + syn_X_clean, dtype=np.float32)
y = np.array(real_y_clean + syn_y_clean)
print(f'Total: {len(X)} ({sum(y)} occupied, {len(y)-sum(y)} empty)')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

clf = SGDClassifier(loss='perceptron', eta0=0.01, learning_rate='constant',
                    max_iter=1000, random_state=42, tol=1e-4)
clf.fit(X_train_s, y_train)

acc = accuracy_score(y_test, clf.predict(X_test_s))
print(f'Test accuracy: {acc*100:.1f}%')
print('Confusion matrix:')
print(confusion_matrix(y_test, clf.predict(X_test_s)))

NameError: name 'syn_X' is not defined

In [ ]:
# Cell 4: Verilog Weights (ROM) and INT8 Quantization
w_scaled  = clf.coef_[0]
w_denorm  = w_scaled / scaler.scale_
max_abs   = np.max(np.abs(w_denorm))
w_int8    = np.clip(np.round(w_denorm / max_abs * 127), -128, 127).astype(int)
bias_raw  = clf.intercept_[0] - np.dot(w_scaled, scaler.mean_ / scaler.scale_)
bias_int  = int(np.clip(np.round(bias_raw / max_abs * 127), -32768, 32767))

# Verify quantized accuracy
correct = 0
for x, lbl in zip(X_test, y_test):
    acc = bias_int
    for i, b in enumerate(x):
        acc += int(b) * int(w_int8[i])
    if (1 if acc > 0 else 0) == lbl:
        correct += 1
print(f'Quantized int8 accuracy: {correct/len(y_test)*100:.1f}%')
print()

# Print Verilog case statement
print('='*50)
print('COPY THIS INTO get_weight() in perceptron_core.v')
print('='*50)
print('case(idx)')
for i, w in enumerate(w_int8):
    sign = '' if w >= 0 else '-'
    print(f"    6'd{i}: get_weight = {sign}8'sd{abs(w)};")
print("    default: get_weight = 8'sd0;")
print('endcase')
print()
print('='*50)
print('COPY THIS AS THE BIAS in reset and start blocks')
print('='*50)
print(f'accumulator <= {bias_int};')

In [ ]:
# Cell 5: Weight Map
plt.figure(figsize=(6, 5))
plt.imshow(w_int8.reshape(8, 8), cmap='RdBu', vmin=-128, vmax=127)
plt.colorbar(label='Weight value')
plt.title('Perceptron weight map (8x8)\nRed=human heat signature  Blue=background')
plt.show()
print('Red pixels = where the model expects a person to be warm')
print('Blue pixels = where the model expects background/cold')